[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/onnx/tutorials/blob/main/08_Model_Optimization_and_Quantization/04_Benchmarking/Benchmarking_Deep_Dive.ipynb)

# 8.4 Benchmarking — Deep Dive

Rigorous benchmarking methodology for ONNX model inference: formal metrics,
warmup detection, latency distributions, statistical significance testing,
throughput modeling, and reproducible comparison frameworks.

| # | Section | Topics |
|---|---------|--------|
| 1 | [Metrics Formalization](#section-1) | Latency, throughput, memory, model size |
| 2 | [Benchmarking Methodology](#section-2) | Warmup, timing, controlled variables, environment isolation |
| 3 | [Percentile Analysis](#section-3) | P50–P99.9, tail amplification in microservices |
| 4 | [Statistical Significance](#section-4) | CI, paired t-test, Cohen's d |
| 5 | [Power Analysis](#section-5) | Minimum detectable effect, sample sizing |
| 6 | [Common Pitfalls](#section-6) | Anti-patterns that invalidate benchmarks |
| 7 | [Throughput Modeling](#section-7) | Batch scaling, Little's Law |
| 8 | [Framework & Reporting](#section-8) | Harness, A/B comparison, multi-config sweep |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats as sp_stats
import time, sys

np.random.seed(42)
plt.rcParams.update({"figure.dpi": 120, "axes.grid": True,
                     "axes.spines.top": False, "axes.spines.right": False})
print(f"Python {sys.version.split()[0]} | NumPy {np.__version__}")

<a id='section-1'></a>
## 1. Metrics Formalization

```
┌─────────────────────────────────────────────────────────────────────────┐
│                      BENCHMARKING PIPELINE                            │
│  ┌──────────┐   ┌──────────┐   ┌──────────┐   ┌──────────┐          │
│  │ PREPARE  │──▶│ WARMUP   │──▶│ MEASURE  │──▶│ ANALYZE  │──▶REPORT │
│  │• Pin freq│   │• JIT     │   │• ≥100    │   │• CI      │  • Table │
│  │• Fix env │   │• Caches  │   │  iters   │   │• t-test  │  • Plots │
│  │• Threads │   │• Discard │   │• perf_   │   │• Cohen d │  • CDF   │
│  └──────────┘   └──────────┘   │  counter │   │• P50–P99 │          │
│                                └──────────┘   └──────────┘          │
└─────────────────────────────────────────────────────────────────────────┘
```

### Latency

Per-request wall-clock time: $T_{\text{latency}} = t_{\text{end}} - t_{\text{start}}$

Use `time.perf_counter()` (monotonic, ~100 ns resolution) — **never** `time.time()`
which is subject to NTP adjustments.

### Throughput

$$\text{Throughput} = \frac{N_{\text{samples}}}{t_{\text{total}}} \quad [\text{samples/sec}]$$

For batched inference with batch size $B$ and per-batch latency $T(B)$:
$\text{Throughput}(B) = B \,/\, T(B)$

### Memory & Model Size

- **Peak RSS**: maximum physical memory via `resource.getrusage`
- **VRAM**: GPU memory peak via `pynvml` or `nvidia-smi`
- **Disk**: $|\text{model.onnx}|$ in bytes — relevant for edge deployment

In [ ]:
def measure_latency(fn, n_iter=200):
    """Collect per-iteration latencies in ms using perf_counter."""
    lats = []
    for _ in range(n_iter):
        t0 = time.perf_counter()
        fn()
        lats.append((time.perf_counter() - t0) * 1000)
    return np.array(lats)

def measure_throughput(fn, batch_size, duration_sec=2.0):
    """Sustained throughput over a fixed time window."""
    count, t0 = 0, time.perf_counter()
    while (time.perf_counter() - t0) < duration_sec:
        fn()
        count += batch_size
    return count / (time.perf_counter() - t0)

def workload(n=64):
    A = np.random.randn(n, n).astype(np.float32)
    return A @ A.T

lats = measure_latency(workload, n_iter=100)
tp = measure_throughput(workload, batch_size=1, duration_sec=1.0)
print(f"Latency  — mean: {np.mean(lats):.3f} ms, std: {np.std(lats):.3f} ms")
print(f"Throughput — {tp:.0f} inferences/sec")

<a id='section-2'></a>
## 2. Benchmarking Methodology

### Warmup Protocol

Early iterations are biased by JIT compilation, cache cold-starts, and
page faults. Latency converges exponentially:

$$T_n = T_\infty + (T_0 - T_\infty)\, e^{-n/\tau}$$

```
Latency
  T₀ ┤ ●
     │  ●
     │   ●●
     │     ●●●
     │        ●●●●●
T_∞  ┤ ─ ─ ─ ─ ─ ─●●●●●●●●●●●●●●●●●  (steady state)
     └────────┼─────────────────────────
              τ                    iteration
     |← warmup →|←── timed region ──→|
```

Steady state detected via **Coefficient of Variation**: $CV = \sigma / \mu < 0.05$.

### Timing: `perf_counter` vs `time.time`

| Timer | Resolution | Monotonic | NTP-immune |
|-------|-----------|-----------|------------|
| `time.perf_counter()` | ~100 ns | Yes | Yes |
| `time.time()` | ~1 µs | No | No |

### Controlled Variables

| Variable | Impact |
|----------|--------|
| Batch size | Latency/throughput scale differently |
| Thread count | Intra/inter-op parallelism |
| Execution Provider | CPU vs CUDA vs TRT ceilings |
| Input distribution | Data-dependent ops, dynamic shapes |

### Environment Isolation

Pin CPU frequency (`cpupower frequency-set -g performance`), disable
turbo boost, close background processes, use dedicated machines.

In [ ]:
def simulate_warmup(n_total=300, T0=18.0, T_inf=5.0, tau=12.0, noise=0.3):
    n = np.arange(n_total)
    lat = T_inf + (T0 - T_inf) * np.exp(-n / tau) + np.random.randn(n_total) * noise
    return np.maximum(lat, 0.1)

def detect_warmup(latencies, window=20, cv_threshold=0.05):
    """Find iteration where CV of sliding window drops below threshold."""
    for i in range(window, len(latencies)):
        chunk = latencies[i - window:i]
        if np.std(chunk) / (np.mean(chunk) + 1e-12) < cv_threshold:
            return i - window
    return len(latencies) // 2

latencies_sim = simulate_warmup()
warmup_end = detect_warmup(latencies_sim)
steady = latencies_sim[warmup_end:]

cvs = [np.std(latencies_sim[i-20:i]) / np.mean(latencies_sim[i-20:i])
       for i in range(20, len(latencies_sim))]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
ax1.plot(latencies_sim, ".", markersize=3, alpha=0.5)
w = 15
ma = np.convolve(latencies_sim, np.ones(w)/w, mode="valid")
ax1.plot(np.arange(w-1, len(latencies_sim)), ma, "r-", lw=2, label="Moving avg")
ax1.axvline(warmup_end, color="red", lw=2, ls="--", label=f"Warmup end = {warmup_end}")
ax1.set_ylabel("Latency (ms)"); ax1.set_title("Warmup Convergence"); ax1.legend()

ax2.plot(range(20, len(latencies_sim)), cvs, lw=1.5)
ax2.axhline(0.05, color="red", ls="--", label="CV = 0.05")
ax2.axvline(warmup_end, color="red", lw=2, ls="--")
ax2.set_xlabel("Iteration"); ax2.set_ylabel("CV"); ax2.legend()
plt.tight_layout(); plt.show()
print(f"Warmup: {warmup_end} iters | Steady: {len(steady)} samples | CV: {np.std(steady)/np.mean(steady):.4f}")

<a id='section-3'></a>
## 3. Percentile Analysis

### Why Mean Is Misleading

Latency distributions are **heavy-tailed** (right-skewed). The mean is
pulled by outliers and misrepresents the typical user experience.

```
  Count
   │   ┌──┐
   │  ┌┤  ├┐
   │ ┌┤│  │├┐
   │ │││  │││    ╭─── long tail (P95, P99, P99.9)
   │ │││  │││   ╭╯
   │ │││  ││├──╯
   └─┴┴┴──┴┴┴────────── Latency (ms)
        ↑
       P50
```

For $n$ sorted samples: $P_k = x_{\lceil k \cdot n / 100 \rceil}$

| Percentile | Meaning | SLA Use |
|------------|---------|--------|
| P50 | Typical request | — |
| P95 | 1-in-20 slower | Common target |
| P99 | 1-in-100 slower | Strict SLA |
| P99.9 | 1-in-1000 slower | Financial/safety |

### Tail Latency Amplification

A user request fans out to $k$ backend calls. User latency = max of $k$
independent calls. Even with P99 = 10 ms per call, with $k = 50$:

$$P(\text{all} < \text{P99}) = 0.99^{50} \approx 0.605$$

So ~40% of user requests hit the tail — making tail optimization critical.

In [ ]:
p50, p95, p99 = [np.percentile(steady, p) for p in (50, 95, 99)]
p999 = np.percentile(steady, 99.9)

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

axes[0].hist(steady, bins=50, density=True, alpha=0.7, color="steelblue", edgecolor="white")
for v, l, c in [(p50,"P50","green"),(p95,"P95","orange"),(p99,"P99","red")]:
    axes[0].axvline(v, color=c, lw=2, ls="--", label=f"{l}={v:.2f}")
axes[0].set_xlabel("Latency (ms)"); axes[0].set_title("PDF"); axes[0].legend(fontsize=8)

ss = np.sort(steady)
cdf = np.arange(1, len(ss)+1) / len(ss)
axes[1].plot(ss, cdf*100, lw=2, color="steelblue")
for v, pct, c in [(p50,50,"green"),(p95,95,"orange"),(p99,99,"red")]:
    axes[1].axhline(pct, color=c, ls=":", alpha=0.5)
    axes[1].axvline(v, color=c, ls="--", lw=1.5, label=f"P{pct}={v:.2f}")
axes[1].set_xlabel("Latency (ms)"); axes[1].set_ylabel("%"); axes[1].set_title("CDF")
axes[1].legend(fontsize=8)

fan_outs = [1, 5, 10, 20, 50, 100]
tail_prob = [1 - 0.99**k for k in fan_outs]
bars = axes[2].bar(range(len(fan_outs)), [p*100 for p in tail_prob],
                   color="#C44E52", alpha=0.8)
axes[2].set_xticks(range(len(fan_outs)))
axes[2].set_xticklabels([str(k) for k in fan_outs])
axes[2].set_xlabel("Fan-out k"); axes[2].set_ylabel("P(hit tail) %")
axes[2].set_title("Tail Amplification")
for bar, p in zip(bars, tail_prob):
    axes[2].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                f"{p*100:.1f}%", ha="center", fontsize=9)

plt.suptitle("Percentile Analysis & Tail Amplification", fontsize=14)
plt.tight_layout(); plt.show()
print(f"P50={p50:.3f}  P95={p95:.3f}  P99={p99:.3f}  P99.9={p999:.3f} ms")

<a id='section-4'></a>
## 4. Statistical Significance

Benchmark numbers are **random variables**. Claiming one config is faster
without statistical testing is scientifically unsound.

### Confidence Interval

$$\bar{x} \pm z_{\alpha/2} \cdot \frac{s}{\sqrt{n}} \qquad \text{(large } n\text{)}$$

$$\bar{x} \pm t_{\alpha/2,\, n-1} \cdot \frac{s}{\sqrt{n}} \qquad \text{(small } n\text{, preferred)}$$

CI width $\propto 1/\sqrt{n}$: 4× more samples halves the width.

### Paired t-Test

When both configs run the **same inputs**, let $d_i = x_{A,i} - x_{B,i}$:

$$t = \frac{\bar{d}}{s_d / \sqrt{n}}$$

Reject $H_0: \bar{d}=0$ if $p < \alpha$ (typically 0.05).

### Cohen's d (Effect Size)

$$d = \frac{\bar{x}_A - \bar{x}_B}{s_{\text{pooled}}} \quad\text{where }\; s_{\text{pooled}} = \sqrt{\frac{(n_A-1)s_A^2 + (n_B-1)s_B^2}{n_A+n_B-2}}$$

| $|d|$ | Interpretation |
|-------|---------------|
| < 0.2 | Negligible |
| 0.2–0.5 | Small |
| 0.5–0.8 | Medium |
| > 0.8 | Large |

In [ ]:
def confidence_interval(data, confidence=0.95):
    n = len(data)
    se = np.std(data, ddof=1) / np.sqrt(n)
    margin = sp_stats.t.ppf((1 + confidence) / 2, df=n-1) * se
    return np.mean(data), margin

sample_sizes = [10, 25, 50, 100, 200, 500]
means_ci, margins_ci = zip(*[confidence_interval(steady[:ns] if ns <= len(steady)
                              else steady) for ns in sample_sizes])

fig, ax = plt.subplots(figsize=(10, 5))
ax.errorbar(range(len(sample_sizes)), means_ci, yerr=margins_ci,
            fmt="o-", capsize=6, lw=2, markersize=8, color="steelblue")
ax.set_xticks(range(len(sample_sizes)))
ax.set_xticklabels([str(n) for n in sample_sizes])
ax.set_xlabel("Number of Samples")
ax.set_ylabel("Mean Latency (ms)")
ax.set_title("95% CI Width Shrinks as $1/\\sqrt{n}$")
ax.axhline(np.mean(steady), color="green", ls="--", alpha=0.5,
           label=f"Full-sample mean = {np.mean(steady):.2f} ms")
ax.legend()
plt.tight_layout(); plt.show()

for ns, m, mg in zip(sample_sizes, means_ci, margins_ci):
    print(f"n={ns:>4d}: {m:.3f} ± {mg:.3f} ms  (width {2*mg:.3f})")

In [ ]:
def cohens_d(x, y):
    nx, ny = len(x), len(y)
    sp = np.sqrt(((nx-1)*np.var(x, ddof=1) + (ny-1)*np.var(y, ddof=1)) / (nx+ny-2))
    return (np.mean(x) - np.mean(y)) / sp

def paired_ttest(x, y):
    d = x - y
    t_stat = np.mean(d) / (np.std(d, ddof=1) / np.sqrt(len(d)))
    p_val = 2 * sp_stats.t.sf(abs(t_stat), df=len(d)-1)
    return t_stat, p_val

np.random.seed(123)
shared = np.random.randn(200) * 0.15
cfg_a = 5.0 + np.random.randn(200) * 0.3 + shared
cfg_b = 4.7 + np.random.randn(200) * 0.3 + shared  # real speedup
cfg_c = 5.0 + np.random.randn(200) * 0.28 + shared  # same speed

print(f"{'Comparison':<22s} {'Diff':>6s} {'t':>7s} {'p':>9s} {'d':>6s} {'Sig?'}")
print("-" * 60)
for name, x, y in [("A vs B (real)", cfg_a, cfg_b),
                    ("A vs C (null)", cfg_a, cfg_c)]:
    t, p = paired_ttest(x, y)
    d = cohens_d(x, y)
    print(f"{name:<22s} {np.mean(x)-np.mean(y):>+6.3f} {t:>7.2f} {p:>9.6f} {d:>6.3f} "
          f"{'YES' if p < 0.05 else 'NO'}")

<a id='section-5'></a>
## 5. Power Analysis — Minimum Detectable Effect

Before benchmarking, determine how many samples detect a given speedup
with power $\geq 0.80$. Required $n$ per group:

$$n \approx \frac{2(z_{\alpha/2} + z_\beta)^2}{d^2}$$

```
          Systematic Tuning Decision Tree

           Is speedup > 10%?
           /             \
        YES               NO
       /                    \
  n ≈ 30–50            Is noise < 5% CV?
  (easy detect)        /             \
                     YES               NO
                    /                    \
             n ≈ 100–200          Fix environment first
             (tight CI)           then re-run n ≈ 200+
```

In [ ]:
def required_n(effect_d, alpha=0.05, power=0.80):
    za = sp_stats.norm.ppf(1 - alpha/2)
    zb = sp_stats.norm.ppf(power)
    return int(np.ceil(2 * (za + zb)**2 / effect_d**2))

def simulate_power(effect_d, n, alpha=0.05, n_sim=2000):
    rej = sum(sp_stats.ttest_ind(np.random.randn(n),
              np.random.randn(n) + effect_d).pvalue < alpha
              for _ in range(n_sim))
    return rej / n_sim

effects = [0.1, 0.2, 0.3, 0.5, 0.8, 1.0]
print(f"{'d':>5s}  {'n needed':>8s}  {'Size'}")
for d in effects:
    label = "negl." if d<0.2 else "small" if d<0.5 else "medium" if d<0.8 else "large"
    print(f"{d:>5.1f}  {required_n(d):>8d}  {label}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.bar(range(len(effects)), [required_n(d) for d in effects], color="steelblue", alpha=0.8)
ax1.set_xticks(range(len(effects))); ax1.set_xticklabels([f"d={d}" for d in effects])
ax1.set_ylabel("n per group"); ax1.set_title("Sample Size for 80% Power")

n_range = [10, 20, 50, 100, 200, 500]
for d in [0.2, 0.5, 0.8]:
    ax2.plot(n_range, [simulate_power(d, n, n_sim=1000) for n in n_range],
            "o-", lw=2, label=f"d={d}")
ax2.axhline(0.80, color="red", ls="--", alpha=0.5, label="Power=0.80")
ax2.set_xlabel("n per group"); ax2.set_ylabel("Power"); ax2.set_ylim(0, 1.05)
ax2.set_title("Power Curves"); ax2.legend(fontsize=9)
plt.tight_layout(); plt.show()

<a id='section-6'></a>
## 6. Common Pitfalls and Anti-Patterns

| # | Trap | Why It Biases Results | Fix |
|---|------|----------------------|-----|
| 1 | Timing first run | Captures JIT overhead | Warmup ≥ 20 iters |
| 2 | Mean without stdev | Hides variability | Report P50, P95, P99 |
| 3 | Laptop on battery | Frequency throttling | Pin AC + fixed freq |
| 4 | Ignoring thermal throttling | Performance degrades over time | Monitor temp |
| 5 | Preprocessing in loop | Misattributes bottleneck | Separate data prep |
| 6 | Insufficient samples | Wide CI | ≥ 100 steady-state |

```
A/B Comparison Visualization Pattern

          Config A (baseline)         Config B ("optimized")
        ┌─────────────────────┐     ┌─────────────────────┐
        │  ●●●●●●●●●●●●●     │     │  ●●●●●●●●●●         │
        │  (tight Gaussian)   │     │           ●● (tail!) │
        │  mean = 5.0 ms      │     │  mean = 5.0 ms       │
        │  P99  = 5.9 ms      │     │  P99  = 11.2 ms      │
        └─────────────────────┘     └─────────────────────┘
        Same mean — very different tail!  Always report percentiles.
```

In [ ]:
np.random.seed(77)
dist_tight = np.random.normal(5.0, 0.3, 500)
dist_heavy = np.concatenate([np.random.normal(4.6, 0.2, 480),
                             np.random.normal(10.0, 1.5, 20)])

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
for ax, data, name in [(axes[0], dist_tight, "A: Tight"),
                        (axes[1], dist_heavy, "B: Heavy Tail")]:
    ax.hist(data, bins=40, density=True, alpha=0.7, color="steelblue", edgecolor="white")
    ax.axvline(np.mean(data), color="red", lw=2, label=f"Mean={np.mean(data):.2f}")
    ax.axvline(np.percentile(data, 99), color="orange", lw=2, ls="--",
               label=f"P99={np.percentile(data,99):.2f}")
    ax.set_title(name); ax.set_xlabel("Latency (ms)"); ax.legend(fontsize=9)

bp = axes[2].boxplot([dist_tight, dist_heavy], labels=["A: Tight", "B: Heavy"],
                     patch_artist=True, widths=0.5)
for patch, c in zip(bp["boxes"], ["#55A868", "#C44E52"]):
    patch.set_facecolor(c); patch.set_alpha(0.7)
axes[2].set_ylabel("Latency (ms)"); axes[2].set_title("Box Plot")
plt.suptitle('The "Mean Lie" — Same Mean, Different Tails', fontsize=14)
plt.tight_layout(); plt.show()

<a id='section-7'></a>
## 7. Throughput Modeling

### Batch-Size Scaling

$$\text{Throughput}(B) = \frac{B}{T(B)}$$

$T(B)$ grows sub-linearly at small $B$ (GPU under-utilized) and linearly
at large $B$ (compute-bound). The optimal batch size maximizes throughput
subject to $T(B) \leq L_{\max}$.

### Little's Law

$$L = \lambda \cdot W$$

- $L$ = avg concurrency (items in system)
- $\lambda$ = arrival rate (req/s = throughput)
- $W$ = avg time in system (latency)

To sustain $\lambda = 100$ req/s at $W = 50$ ms latency:
$L = 100 \times 0.05 = 5$ concurrent workers.

In [ ]:
batch_sizes = [1, 2, 4, 8, 16, 32, 64, 128]
np.random.seed(99)
lats_b = [max(2.0 + 0.8*np.log2(b+1) + 0.001*b**1.3 + np.random.randn()*0.1, 0.5)
          for b in batch_sizes]
tps_b = [b / (l/1000) for b, l in zip(batch_sizes, lats_b)]
best_idx = int(np.argmax(tps_b))

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

axes[0].plot(batch_sizes, lats_b, "o-", lw=2, color="steelblue", markersize=7)
axes[0].set_xlabel("Batch Size"); axes[0].set_ylabel("Latency (ms)")
axes[0].set_title("Latency vs Batch Size"); axes[0].set_xscale("log", base=2)

axes[1].plot(batch_sizes, tps_b, "s-", lw=2, color="#55A868", markersize=7)
axes[1].axvline(batch_sizes[best_idx], color="red", ls="--",
               label=f"Best BS={batch_sizes[best_idx]}")
axes[1].set_xlabel("Batch Size"); axes[1].set_ylabel("Throughput (samp/s)")
axes[1].set_title("Throughput vs Batch Size"); axes[1].set_xscale("log", base=2)
axes[1].legend()

target_rps = [50, 100, 200, 500, 1000]
W = 0.005  # 5 ms
conc = [lam * W for lam in target_rps]
axes[2].bar(range(len(target_rps)), conc, color="#DD8452", alpha=0.8)
axes[2].set_xticks(range(len(target_rps)))
axes[2].set_xticklabels([f"{r}" for r in target_rps])
axes[2].set_xlabel("Target λ (req/s)"); axes[2].set_ylabel("Concurrency L")
axes[2].set_title(f"Little's Law: L = λ·W  (W={W*1000:.0f}ms)")
for i, c in enumerate(conc):
    axes[2].text(i, c+0.1, f"{c:.1f}", ha="center", fontsize=10)

plt.suptitle("Throughput Modeling & Little's Law", fontsize=14)
plt.tight_layout(); plt.show()

print(f"{'BS':>4s}  {'Lat(ms)':>8s}  {'TP(s/s)':>8s}")
for b, l, t in zip(batch_sizes, lats_b, tps_b):
    print(f"{b:>4d}  {l:>8.2f}  {t:>8.0f}{' ◀ best' if b==batch_sizes[best_idx] else ''}")

<a id='section-8'></a>
## 8. Benchmarking Framework, A/B Comparison & Multi-Config Sweep

```
benchmark(run_fn, config)
  ├── 1. Warmup iterations (discard)
  ├── 2. Timed loop → per-iter latencies via perf_counter
  ├── 3. CV-based warmup detection
  ├── 4. Stats: mean, CI, P50–P99.9, throughput, CV
  └── 5. Return BenchmarkResult
```

In [ ]:
class BenchmarkResult:
    def __init__(self, name, raw, warmup_end, batch_size=1):
        self.name, self.raw, self.warmup_end = name, raw, warmup_end
        self.steady = raw[warmup_end:]
        self.batch_size = batch_size
        self.mean = np.mean(self.steady)
        self.std = np.std(self.steady, ddof=1)
        self.cv = self.std / self.mean
        self.p50 = np.percentile(self.steady, 50)
        self.p95 = np.percentile(self.steady, 95)
        self.p99 = np.percentile(self.steady, 99)
        n = len(self.steady)
        self.ci = sp_stats.t.ppf(0.975, df=n-1) * self.std / np.sqrt(n)
        self.throughput = batch_size / (self.mean / 1000.0)

    def summary(self):
        return (f"{self.name}: {self.mean:.3f}±{self.ci:.3f} ms  "
                f"P50={self.p50:.3f}  P95={self.p95:.3f}  P99={self.p99:.3f}  "
                f"CV={self.cv:.4f}  TP={self.throughput:.0f}/s")

def benchmark(run_fn, name="bench", n_warmup=50, n_iter=500, batch_size=1):
    for _ in range(n_warmup):
        run_fn()
    lats = []
    for _ in range(n_iter):
        t0 = time.perf_counter()
        run_fn()
        lats.append((time.perf_counter() - t0) * 1000)
    lats = np.array(lats)
    return BenchmarkResult(name, lats, detect_warmup(lats), batch_size)

r = benchmark(lambda: workload(64), name="MatMul-64", n_warmup=30, n_iter=300)
print(r.summary())

In [ ]:
def compare_ab(ra, rb):
    n = min(len(ra.steady), len(rb.steady))
    a, b = ra.steady[:n], rb.steady[:n]
    t, p = paired_ttest(a, b)
    d = cohens_d(a, b)
    return {"diff": np.mean(a)-np.mean(b), "pct": (np.mean(a)-np.mean(b))/np.mean(a)*100,
            "t": t, "p": p, "d": d, "sig": p < 0.05,
            "label": "negl." if abs(d)<0.2 else "small" if abs(d)<0.5
                     else "medium" if abs(d)<0.8 else "large"}

r_base = benchmark(lambda: workload(64), "Baseline-64", n_warmup=30, n_iter=300)
r_opt  = benchmark(lambda: workload(48), "Optimized-48", n_warmup=30, n_iter=300)
r_same = benchmark(lambda: workload(64), "Same-64-v2", n_warmup=30, n_iter=300)

for label, rb in [("Base vs Opt", r_opt), ("Base vs Same", r_same)]:
    c = compare_ab(r_base, rb)
    sig = "SIGNIFICANT" if c["sig"] else "not sig."
    print(f"{label}: diff={c['diff']:+.3f}ms ({c['pct']:+.1f}%)  "
          f"t={c['t']:.2f}  p={c['p']:.6f}  d={c['d']:.3f}({c['label']})  {sig}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
data_box = [r_base.steady, r_opt.steady, r_same.steady]
bp = ax1.boxplot(data_box, labels=[r.name for r in [r_base, r_opt, r_same]],
                patch_artist=True, widths=0.5)
for patch, c in zip(bp["boxes"], ["#4C72B0", "#55A868", "#DD8452"]):
    patch.set_facecolor(c); patch.set_alpha(0.7)
ax1.set_ylabel("Latency (ms)"); ax1.set_title("A/B Box Plot")

for res, c in [(r_base,"#4C72B0"), (r_opt,"#55A868"), (r_same,"#DD8452")]:
    s = np.sort(res.steady); cd = np.arange(1,len(s)+1)/len(s)
    ax2.plot(s, cd*100, lw=2, label=res.name, color=c)
ax2.set_xlabel("Latency (ms)"); ax2.set_ylabel("%"); ax2.set_title("CDF")
ax2.legend(fontsize=9)
plt.suptitle("A/B Comparison with Significance Testing", fontsize=14)
plt.tight_layout(); plt.show()

In [ ]:
sizes = [32, 64, 128, 256]
sweep = [benchmark(lambda n=sz: workload(n), f"MatMul-{sz}", 20, 200) for sz in sizes]

print(f"{'Config':<14s} {'Mean±CI':>14s} {'P50':>7s} {'P95':>7s} {'P99':>7s} "
      f"{'CV':>6s} {'TP/s':>7s}")
print("-" * 70)
for r in sweep:
    print(f"{r.name:<14s} {r.mean:>6.3f}±{r.ci:<6.3f} {r.p50:>7.3f} {r.p95:>7.3f} "
          f"{r.p99:>7.3f} {r.cv:>6.4f} {r.throughput:>7.0f}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
colors = ["#4C72B0", "#55A868", "#DD8452", "#C44E52"]

axes[0,0].errorbar(sizes, [r.mean for r in sweep], yerr=[r.ci for r in sweep],
                   fmt="o-", capsize=5, lw=2, markersize=7, color="steelblue",
                   label="Mean±CI")
axes[0,0].plot(sizes, [r.p95 for r in sweep], "s--", color="orange", label="P95")
axes[0,0].plot(sizes, [r.p99 for r in sweep], "^--", color="red", label="P99")
axes[0,0].set_xlabel("Size"); axes[0,0].set_ylabel("Latency (ms)")
axes[0,0].set_title("Latency vs Problem Size"); axes[0,0].legend(fontsize=9)

axes[0,1].barh(range(len(sizes)), [r.throughput for r in sweep], color=colors, alpha=0.8)
axes[0,1].set_yticks(range(len(sizes)))
axes[0,1].set_yticklabels([r.name for r in sweep], fontsize=9)
axes[0,1].set_xlabel("Throughput (samp/s)"); axes[0,1].set_title("Throughput")

for r, c in zip(sweep, colors):
    s = np.sort(r.steady); cd = np.arange(1,len(s)+1)/len(s)
    axes[1,0].plot(s, cd*100, lw=2, label=r.name, color=c)
axes[1,0].set_xlabel("Latency (ms)"); axes[1,0].set_ylabel("%")
axes[1,0].set_title("CDF Overlay"); axes[1,0].legend(fontsize=8)

axes[1,1].bar(range(len(sizes)), [r.cv for r in sweep], color="steelblue", alpha=0.8)
axes[1,1].axhline(0.05, color="red", ls="--", label="CV=5%")
axes[1,1].axhline(0.01, color="green", ls="--", label="CV=1%")
axes[1,1].set_xticks(range(len(sizes)))
axes[1,1].set_xticklabels([f"{sz}×{sz}" for sz in sizes])
axes[1,1].set_ylabel("CV"); axes[1,1].set_title("Measurement Stability")
axes[1,1].legend(fontsize=9)

plt.suptitle("Multi-Configuration Benchmark Dashboard", fontsize=15)
plt.tight_layout(); plt.show()

print("\nPairwise vs Baseline:")
for r in sweep[1:]:
    c = compare_ab(sweep[0], r)
    print(f"  {r.name}: diff={c['diff']:+.3f}ms  p={c['p']:.6f}  d={c['d']:.3f}  "
          f"{'SIG' if c['sig'] else 'n.s.'}")

## Key Takeaways

1. **Formalize metrics** — latency ($T = t_{\text{end}} - t_{\text{start}}$),
   throughput ($N / t_{\text{total}}$), memory, model size
2. **Always warm up** — discard early iterations; detect steady state via CV < 5%
3. **Report distributions** — P50, P95, P99, P99.9 reveal tails that the mean hides
4. **Tail latency amplifies** — with $k$ fan-out: $P(\text{any hit}) = 1-(1-p)^k$
5. **Confidence intervals**: $\bar{x} \pm t_{\alpha/2} \cdot s/\sqrt{n}$ — width $\propto 1/\sqrt{n}$
6. **Statistical tests required** — paired $t$-test + Cohen's $d$ for practical significance
7. **Power analysis first** — small effects ($d<0.2$) need $n>700$; plan accordingly
8. **Little's Law**: $L = \lambda \cdot W$ links concurrency, throughput, and latency
9. **Avoid pitfalls** — timing cold runs, mean-only reporting, preprocessing in loop
10. **Automate everything** — reusable harness ensures reproducibility